# Efficient VLM Inference via Visual Token Compression

**Authors:** Haojing Tong (ht2667), Zhuoao Wang (zw4721), Yuqi Wang (yw4338)

This notebook runs the VLM visual token compression benchmark on Google Colab.

**Before running:** Go to `Runtime -> Change runtime type -> T4 GPU` (or A100 if available with Colab Pro).

## 1. Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")

## 2. Clone the Project

Option A: Clone from GitHub (recommended)

In [ ]:
# TODO: replace with your own GitHub repo URL
# !git clone https://github.com/<your-username>/Project.git
# %cd Project

Option B: Upload the project zip manually via the Colab file browser, then unzip.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # upload Project.zip
# !unzip -q Project.zip
# %cd Project

Option C: Mount Google Drive if the project is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: change the path to where you put the project on Drive
%cd /content/drive/MyDrive/HPML/Project

In [ ]:
!pwd && ls

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Install the project as an editable package so `from src.xxx import ...` works
!pip install -q -e .

## 4. Smoke Test: Compression Modules

Verify the three compression strategies produce the expected output shapes using random tensors.

In [ ]:
import torch
from src.compression import FixedRatioPruner, ImportanceBasedPruner, TokenMerger

# Fake visual tokens: batch=1, 100 tokens, hidden_dim=768
tokens = torch.randn(1, 100, 768)
cfg = {"retention_ratio": 0.5}

print("FixedRatioPruner output shape:     ", FixedRatioPruner(cfg).compress(tokens).shape)
print("ImportanceBasedPruner output shape:", ImportanceBasedPruner(cfg).compress(tokens).shape)
print("TokenMerger output shape:          ", TokenMerger(cfg).compress(tokens).shape)

## 5. Load Qwen2.5-VL Model

Start with the 3B version. If you have Colab Pro + A100, you can try 7B.

In [ ]:
import yaml

with open("configs/default.yaml") as f:
    config = yaml.safe_load(f)

# Override for Colab: use 3B and float16
config["model"]["name"] = "Qwen/Qwen2.5-VL-3B-Instruct"
config["model"]["dtype"] = "float16"
config["model"]["device"] = "cuda"

print(config["model"])

In [ ]:
from src.models import load_model

model, processor = load_model(config)
print("Model loaded successfully.")
print(f"Peak GPU memory after load: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

## 6. Single Inference Test

Download a sample image and run one inference to confirm the full pipeline works.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
image

In [ ]:
# Qwen2.5-VL uses a chat-style conversation template
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this image in detail."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=128)

generated = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)[0]
print(generated)

## 7. Baseline Profiling (No Compression)

Measure latency, throughput, and peak memory with full visual tokens.

In [ ]:
from src.utils.profiler import InferenceProfiler

profiler = InferenceProfiler(num_warmup=2, num_runs=5)

def inference_fn():
    with torch.no_grad():
        return model.generate(**inputs, max_new_tokens=64)

metrics_baseline = profiler.profile(inference_fn)
for k, v in metrics_baseline.items():
    print(f"{k}: {v:.2f}")

## 8. Mini Benchmark: Compare Compression Ratios

**Note:** This cell requires the compressor to be hooked into the model forward pass (still a TODO in `evaluator.py`).
For now this is a placeholder that shows the structure of the benchmark grid.

In [ ]:
import itertools
import pandas as pd

methods = ["none", "fixed_ratio", "importance", "token_merging"]
retention_ratios = [1.0, 0.75, 0.5, 0.25, 0.1]

rows = []
for method, ratio in itertools.product(methods, retention_ratios):
    if method == "none" and ratio != 1.0:
        continue
    if method != "none" and ratio == 1.0:
        continue

    # TODO: once compressor hook is implemented, run actual inference here
    # For now, record the configuration only
    rows.append({
        "method": method,
        "retention_ratio": ratio,
        "latency_ms": None,
        "peak_memory_mb": None,
    })

df = pd.DataFrame(rows)
df

## 9. Save Results

Persist results to Google Drive so they survive Colab disconnects.

In [ ]:
import os
os.makedirs("results", exist_ok=True)
df.to_csv("results/benchmark_summary.csv", index=False)
print("Saved to results/benchmark_summary.csv")

## 10. (Optional) Visualization

Placeholder for latency-vs-accuracy trade-off plots once benchmark data is populated.

In [ ]:
import matplotlib.pyplot as plt

# TODO: plot latency vs retention ratio for each compression method
# fig, ax = plt.subplots(figsize=(6, 4))
# for method, group in df.groupby("method"):
#     ax.plot(group["retention_ratio"], group["latency_ms"], marker="o", label=method)
# ax.set_xlabel("Retention Ratio")
# ax.set_ylabel("Latency (ms)")
# ax.legend()
# plt.show()